In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.4 MB/s eta 0:00:00


# Import Libraries

In [2]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV

from utils.models import get_models
from utils.nested_cv import run_outer_fold_loop
from utils.evaluation import evaluate_model, evaluate_outer_fold

from utils.optimization import(
    get_param_dist,
    get_search_iter,
    tune_model
)

from utils.pipeline import(
    create_pipeline,
    save_fold_predictions,
    select_best_threshold
)
import shap

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load dataset and model

In [3]:
SELECTED_MODEL = "LightGBM"

In [4]:
ds = pd.read_csv("/content/DiabeticCKD_engineered.csv")
group_ds = pd.read_csv("/content/Patient_Groups.csv")

group = group_ds["Patient_ID"]
X = ds.drop("CKD", axis=1)
y = ds["CKD"]

In [5]:
models = get_models()
model_name = SELECTED_MODEL
model = models[SELECTED_MODEL]

# Create folders for result

In [6]:
experiment4_dir = Path("/content/exp4_ds2")

(experiment4_dir / "shap_rankings").mkdir(
    parents=True,
    exist_ok=True
)

(experiment4_dir / "reduced_models").mkdir(
    exist_ok=True
)

(experiment4_dir / "fold_results").mkdir(
    exist_ok=True
)

(experiment4_dir / "summary").mkdir(
    exist_ok=True
)

# Cross-validation

In [7]:
outer_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Find the best pipeline for the selected model

In [8]:
def get_pipeline(selected_model, estimator, groups):
    pipeline = create_pipeline(
        selected_model,
        estimator
      )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=get_param_dist(SELECTED_MODEL),
        n_iter=get_search_iter(SELECTED_MODEL),
        scoring="average_precision",
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train, groups=groups)

    best_pipeline = search.best_estimator_

    # print(search.best_score_)
    return best_pipeline

# Find top ranking features

In [9]:
def shap_ranking(X_train, model):
  explainer = shap.TreeExplainer(model)
  shap_values = explainer.shap_values(X_train)

  # Binary classification
  if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # Safety check
  assert shap_values.ndim == 2, (
        f"Unexpected SHAP shape: {shap_values.shape}"
  )

  assert shap_values.shape[1] == X_train.shape[1], (
        f"SHAP features ({shap_values.shape[1]}) "
        f"!= X features ({X_train.shape[1]})"
  )

  mean_abs_shap = np.abs(shap_values).mean(axis=0)

  importance_df = pd.DataFrame({
      "feature": X_train.columns,
      "mean_abs_shap": mean_abs_shap
  })

  importance_df = importance_df.sort_values(
      by="mean_abs_shap",
      ascending=False
  ).reset_index(drop=True)

  importance_df.insert(
      0,
      "rank",
      np.arange(1, len(importance_df) + 1)
  )
  # importance_df["rank"] = np.arange(1, len(importance_df) + 1)
  return importance_df

# Feature ranking for all folds

In [10]:
fold_rankings = []
top10_per_fold = []
top5_per_fold = []

In [11]:
for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, group),
        start=1):

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train = group.iloc[train_idx]
    # groups_test = group.iloc[test_idx]

    ####### get the best pipeline for the selected model #######
    pipeline = get_pipeline(SELECTED_MODEL, model, groups_train)
    model = pipeline.named_steps["model"]

    ####### get top ranking features #######
    importance_df = shap_ranking(X_train, model)

    ####### save feature ranks per fold #######
    importance_df.to_csv(
        experiment4_dir /
        "shap_rankings" /
        f"fold_{fold}_ranking.csv",
        index=False
    )

    fold_rankings.append(importance_df)

    top10_per_fold.append(
        importance_df["feature"].head(10).tolist()
    )

    top5_per_fold.append(
        importance_df["feature"].head(5).tolist()
    )

# Combine all fold rankings

In [12]:
all_rankings = pd.concat(
    fold_rankings,
    keys=range(1, len(fold_rankings) + 1),
    names=["fold"]
).reset_index(level=0)

all_rankings.rename(columns={"level_0": "fold"}, inplace=True)

print(all_rankings.head())

   fold  rank         feature  mean_abs_shap
0     1     1        Age_Diff       1.515507
1     1     2   Diabetic_Year       0.527765
2     1     3  Calorie_Intake       0.256666
3     1     4    BMI_per_Year       0.254601
4     1     5    BMI_Diabetes       0.230907


# Get a stable ranking for average SHAP

In [13]:
stable_ranking = (
    all_rankings
    .groupby("feature")
    .agg(
        mean_shap=("mean_abs_shap", "mean"),
        std_shap=("mean_abs_shap", "std"),
        mean_rank=("rank", "mean")
    )
    .reset_index()
)

stable_ranking = stable_ranking.sort_values(
    by="mean_shap",
    ascending=False
).reset_index(drop=True)

stable_ranking.insert(
    0,
    "final_rank",
    np.arange(1, len(stable_ranking)+1)
)

stable_ranking.to_csv(
    experiment4_dir /
    "summary" /
    "stable_feature_ranking.csv",
    index=False
)

In [14]:
stable_ranking.head()

,final_rank,feature,mean_shap,std_shap,mean_rank
0,1,Age_Diff,3.002058,1.332280,1.0
1,2,Diabetic_Year,1.694655,1.032065,2.0
2,3,Calorie_Intake,0.794799,0.600371,6.8
3,4,Height,0.736810,0.514983,5.2
4,5,BMI_per_Year,0.700725,0.477271,6.8


# Get top 10 and top 5 features

In [15]:
top10_features = (
    stable_ranking["feature"]
    .head(10)
    .tolist()
)

top5_features = (
    stable_ranking["feature"]
    .head(5)
    .tolist()
)

print("Top 10 Features")
print(top10_features)

print("\nTop 5 Features")
print(top5_features)

Top 10 Features
['Age_Diff', 'Diabetic_Year', 'Calorie_Intake', 'Height', 'BMI_per_Year', 'BMI_Diabetes', 'BMI', 'Age', 'Average_Age', 'Average_Weight']

Top 5 Features
['Age_Diff', 'Diabetic_Year', 'Calorie_Intake', 'Height', 'BMI_per_Year']


In [16]:
pd.DataFrame({
    "Top10": pd.Series(top10_features)
}).to_csv(
    experiment4_dir /
    "summary" /
    "top10_features.csv",
    index=False
)

pd.DataFrame({
    "Top5": pd.Series(top5_features)
}).to_csv(
    experiment4_dir /
    "summary" /
    "top5_features.csv",
    index=False
)

In [17]:
X_top10 = X[top10_features].copy()

X_top5 = X[top5_features].copy()

print(X_top10.shape)
print(X_top5.shape)

(4000, 10)
(4000, 5)


# Verify feature stability

In [18]:
from collections import Counter

top10_counts = Counter()

for features in top10_per_fold:
    top10_counts.update(features)

feature_frequency = (
    pd.DataFrame(
        top10_counts.items(),
        columns=["feature", "top10_frequency"]
    )
    .sort_values("top10_frequency", ascending=False)
)

feature_frequency.to_csv(
    experiment4_dir /
    "summary" /
    "top10_feature_frequency.csv",
    index=False
)

print(feature_frequency)

              feature  top10_frequency
0            Age_Diff                5
1       Diabetic_Year                5
5              Height                5
2      Calorie_Intake                4
3        BMI_per_Year                4
4        BMI_Diabetes                4
7        Take_Insulin                4
8         Average_Age                4
11                Age                4
6      Average_Weight                3
10                BMI                3
9   Urinary_Infection                2
13       Age_Diabetes                2
12     Walk_Regularly                1


# Extract Selected Model's Previous Result

In [19]:
exp4_top10 = {}
exp4_top5 = {}
result_rows = []

In [20]:
metric_columns = [
      "PR AUC",
      "ROC AUC",
      "F1",
      "MCC",
      "Sensitivity",
      "Specificity",
      "Balanced Accuracy",
      "Accuracy",
      "Brier Score"
  ]


In [22]:
#### load joblib file #####
exp1_dict = joblib.load("experiment1_dataset2.joblib")
exp1_dict[SELECTED_MODEL].keys()

dict_keys(['fold_metrics', 'best_params', 'thresholds', 'best_inner_mcc', 'saved_folds', 'saved_models', 'fold_results', 'mean', 'std'])

In [23]:
result_dict = {}
mean = exp1_dict[SELECTED_MODEL]['mean']
std = exp1_dict[SELECTED_MODEL]['std']
result_dict["Result"] = "full_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)
for k,v in result_dict.items():
  print(f"{k}: {v}")

Result: full_feat
PR AUC: 0.5202 ± 0.0459
ROC AUC: 0.8858 ± 0.0283
F1: 0.4672 ± 0.0659
MCC: 0.4166 ± 0.0787
Sensitivity: 0.5277 ± 0.1656
Specificity: 0.9196 ± 0.0588
Balanced Accuracy: 0.7237 ± 0.0604
Accuracy: 0.8824 ± 0.0420
Brier Score: 0.0778 ± 0.0133


# Run nested CV for top 10 features

In [24]:
exp4_top10[SELECTED_MODEL] = {
    "fold_metrics": [],
    "best_params": [],
    "thresholds": [],
    "best_inner_mcc": [],
    "saved_folds": [],
    "saved_models": []
}

exp4_top10_dir = Path("/content/exp4_top10")

MODEL_DIR = os.path.join(exp4_top10_dir, "model_dir")
FOLD_DIR = os.path.join(exp4_top10_dir, "fold_dir")

os.makedirs(exp4_top10_dir, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)


In [25]:
 ################# Outer fold loop #################
run_outer_fold_loop(exp4_top10, model_name, model, X_top10, y, outer_cv, inner_cv,
                    FOLD_DIR, MODEL_DIR, groups=group, scoring="average_precision")
################# Add aggregation #################
fold_df = pd.DataFrame(
        exp4_top10[model_name]["fold_metrics"]

  )
fold_df.to_csv(experiment4_dir /"fold_results" /"top10_feat_fold_results.csv",
               index=False, encoding='utf-8-sig')
exp4_top10[model_name]["fold_results"] = fold_df

################# Calculate mean and std for the df #################
metric_df = fold_df[metric_columns]
mean = metric_df.mean()
std = metric_df.std()

exp4_top10[model_name]["mean"] = (
        mean.to_dict()
)
exp4_top10[model_name]["std"] = (
            std.to_dict()
  )

In [26]:
result_dict = {}
result_dict["Result"] = "top10_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)

for k,v in result_dict.items():
  print(f"{k}: {v}")

Result: top10_feat
PR AUC: 0.4984 ± 0.0642
ROC AUC: 0.8877 ± 0.0270
F1: 0.4716 ± 0.0300
MCC: 0.4225 ± 0.0332
Sensitivity: 0.6711 ± 0.0844
Specificity: 0.8723 ± 0.0432
Balanced Accuracy: 0.7717 ± 0.0270
Accuracy: 0.8530 ± 0.0336
Brier Score: 0.0995 ± 0.0141


# Run nested CV for top 5 features

In [27]:
exp4_top5_dir = Path("/content/exp4_top5")

MODEL_DIR = os.path.join(exp4_top5_dir, "model_dir")
FOLD_DIR = os.path.join(exp4_top5_dir, "fold_dir")

os.makedirs(exp4_top5_dir, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FOLD_DIR, exist_ok=True)

exp4_top5[SELECTED_MODEL]  = {
    "fold_metrics": [],
    "best_params": [],
    "thresholds": [],
    "best_inner_mcc": [],
    "saved_folds": [],
    "saved_models": []
}

In [28]:
 ################# Outer fold loop #################
run_outer_fold_loop(exp4_top5, model_name, model, X_top5, y,outer_cv, inner_cv,
                    FOLD_DIR, MODEL_DIR, groups=group, scoring="average_precision")
################# Add aggregation #################
fold_df = pd.DataFrame(
        exp4_top5[model_name]["fold_metrics"]

  )
fold_df.to_csv("top5_feat_fold_results.csv", index=False, encoding='utf-8-sig')
exp4_top5[model_name]["fold_results"] = fold_df

  ################# Calculate mean and std for the df #################
metric_df = fold_df[metric_columns]
mean = metric_df.mean()
std = metric_df.std()

exp4_top5[model_name]["mean"] = (
        mean.to_dict()
)
exp4_top5[model_name]["std"] = (
            std.to_dict()
  )


In [29]:
result_dict = {}
result_dict["Result"] = "top5_feat"
for metric in metric_columns:
    result_dict[metric] = f"{mean[metric]:.4f} ± {std[metric]:.4f}"
result_rows.append(result_dict)
for k,v in result_dict.items():
  print(f"{k}: {v}")


Result: top5_feat
PR AUC: 0.4741 ± 0.0984
ROC AUC: 0.8818 ± 0.0300
F1: 0.4761 ± 0.0705
MCC: 0.4226 ± 0.0798
Sensitivity: 0.6029 ± 0.1462
Specificity: 0.9034 ± 0.0255
Balanced Accuracy: 0.7531 ± 0.0643
Accuracy: 0.8747 ± 0.0172
Brier Score: 0.0980 ± 0.0159


In [30]:
result_df = pd.DataFrame(result_rows)
result_df.to_csv(
        "experiment4_dataset2_result.csv",
        index=False,
        encoding='utf-8-sig'
)
print(result_df)

       Result           PR AUC          ROC AUC               F1  \
0   full_feat  0.5202 ± 0.0459  0.8858 ± 0.0283  0.4672 ± 0.0659   
1  top10_feat  0.4984 ± 0.0642  0.8877 ± 0.0270  0.4716 ± 0.0300   
2   top5_feat  0.4741 ± 0.0984  0.8818 ± 0.0300  0.4761 ± 0.0705   

               MCC      Sensitivity      Specificity Balanced Accuracy  \
0  0.4166 ± 0.0787  0.5277 ± 0.1656  0.9196 ± 0.0588   0.7237 ± 0.0604   
1  0.4225 ± 0.0332  0.6711 ± 0.0844  0.8723 ± 0.0432   0.7717 ± 0.0270   
2  0.4226 ± 0.0798  0.6029 ± 0.1462  0.9034 ± 0.0255   0.7531 ± 0.0643   

          Accuracy      Brier Score  
0  0.8824 ± 0.0420  0.0778 ± 0.0133  
1  0.8530 ± 0.0336  0.0995 ± 0.0141  
2  0.8747 ± 0.0172  0.0980 ± 0.0159  
